# Modelo RandomForestClassifier

En este notebook se entrena un modelo Random Forest para clasificar géneros musicales utilizando la tabla macro generada previamente.

## Problema y objetivos

La empresa quiere identificar el género de una canción a partir de sus características de audio. La variable `id_genero` no es una etiqueta de sí o no, ni un valor numérico a estimar: toma 10 valores distintos y cada canción pertenece a una sola clase. El problema es de **clasificación supervisada multiclase**.

En este notebook el objetivo es entrenar un Random Forest que, con las variables numéricas de la tabla macro, asigne cada fragmento de 3 segundos a una de esas 10 clases y, por voto mayoritario, obtenga también el género de la canción. El modelo se ajusta solo con las canciones de entrenamiento; el conjunto de test se reserva para la evaluación. El resultado se mide con accuracy y F1 macro, para ver el acierto global y el rendimiento medio entre géneros.

## Selección de métricas y umbral

**Accuracy.** Hay 10 géneros y el mismo número de canciones en cada clase. Con ese reparto equilibrado, el porcentaje de aciertos resume el rendimiento global sin que una clase más frecuente lo distorsione.

**F1 macro.** Se calcula el F1 de cada género y se promedia dando el mismo peso a las 10 clases. Así, un género fácil de clasificar no compensa a otro que el modelo confunde. El F1 por clase del informe de clasificación muestra cuáles son esos géneros.

**Umbral de decisión.** El modelo devuelve una probabilidad para cada una de las 10 clases. La clase asignada es la de probabilidad más alta. No se exige una probabilidad mínima para aceptar la predicción: si la clase ganadora tiene, por ejemplo, 0,2 y el resto queda por debajo, esa clase se predice igualmente.

In [ ]:
import pandas as pd

In [ ]:
from pathlib import Path

def carpeta_proyecto(nombre, marca):
    """Localiza los CSV en Jupyter, Visual Studio Code y Colab."""
    from pathlib import Path
    import json

    saltar = {
        ".config", "sample_data", "drive", ".git", "__pycache__",
        "node_modules", ".ipynb_checkpoints", "site-packages",
    }

    def directorio_notebook():
        try:
            return Path(__vsc_ipynb_file__).resolve().parent
        except NameError:
            pass
        try:
            from urllib.request import urlopen
            try:
                from ipykernel.connect import get_connection_file
            except ImportError:
                from ipykernel import get_connection_file
            conexion = Path(get_connection_file())
            kernel_id = conexion.name.replace("kernel-", "").replace(".json", "")
            servidores = list(conexion.parent.glob("jpserver-*.json"))[:3]
            for servidor in servidores:
                try:
                    info = json.loads(servidor.read_text(encoding="utf-8"))
                except (OSError, json.JSONDecodeError):
                    continue
                url = str(info.get("url", "")).rstrip("/")
                raiz = info.get("root_dir")
                if not url or not raiz:
                    continue
                api = url + "/api/sessions"
                token = info.get("token") or ""
                if token:
                    api += "?token=" + token
                try:
                    with urlopen(api, timeout=0.8) as respuesta:
                        sesiones = json.loads(respuesta.read().decode())
                except Exception:
                    continue
                for sesion in sesiones:
                    kid = (sesion.get("kernel") or {}).get("id")
                    if kid != kernel_id:
                        continue
                    cuaderno = sesion.get("notebook") or {}
                    relativo = cuaderno.get("path") or sesion.get("path")
                    if relativo:
                        return (Path(raiz) / relativo).resolve().parent
        except Exception:
            pass
        return None

    bases = []
    notebook = directorio_notebook()
    if notebook is not None:
        bases.append(notebook)
    bases.append(Path.cwd().resolve())
    content = Path("/content")
    if content.is_dir():
        bases.append(content)
        drive = content / "drive" / "MyDrive"
        if drive.is_dir():
            bases.append(drive)

    nombradas = []
    sueltas = []
    vistos = set()

    def considerar(directorio, aceptar_suelto):
        if directorio in vistos:
            return
        vistos.add(directorio)
        try:
            carpeta = directorio / nombre
            if (carpeta / marca).is_file():
                nombradas.append(carpeta)
                return
            if directorio.name == nombre and (directorio / marca).is_file():
                nombradas.append(directorio)
                return
            if aceptar_suelto and (directorio / marca).is_file():
                sueltas.append(directorio)
        except OSError:
            return

    def hijos(directorio, profundidad, limite):
        if profundidad < 0 or limite[0] <= 0 or not directorio.is_dir():
            return
        try:
            entradas = list(directorio.iterdir())
        except OSError:
            return
        for hijo in entradas:
            if not hijo.is_dir() or hijo.name.startswith(".") or hijo.name in saltar:
                continue
            limite[0] -= 1
            considerar(hijo, False)
            if nombradas or limite[0] <= 0:
                return
            hijos(hijo, profundidad - 1, limite)
            if nombradas or limite[0] <= 0:
                return

    for base in bases:
        considerar(base, True)
        if nombradas:
            break
        for padre in list(base.parents)[:6]:
            considerar(padre, False)
            if nombradas:
                break
        if nombradas:
            break
        profundidad = 3 if base.name in {"content", "MyDrive"} else 2
        hijos(base, profundidad, [400])
        if nombradas:
            break

    if nombradas:
        return nombradas[0]
    if sueltas:
        return sueltas[0]
    raise FileNotFoundError(
        "No se encuentra " + marca + ". "
        "Tiene que existir la carpeta '" + nombre + "' en el proyecto, "
        "o el archivo en el directorio de trabajo. En Colab puede estar junto a sample_data."
    )


def carpeta_datos():
    """Carpeta datos preprocesados, en Jupyter, Visual Studio Code o Colab."""
    return carpeta_proyecto("datos preprocesados", "ids_train.csv")

DATOS = carpeta_datos()
print("Datos:", DATOS)

# Cargar datos base
df_macro = pd.read_csv(DATOS / "df_macro.csv")

In [ ]:
# Revisar valores nulos iniciales

nulos_macro = df_macro.isna().sum()

display(nulos_macro[nulos_macro > 0].sort_values(ascending=False))

,0
duracion,180


En la fase de análisis exploratorio se localizaron valores nulos en la variable duración. Tras revisar las canciones afectadas, se comprobó que estos registros correspondían a pistas de 30 segundos, por lo que se añadieron los valores faltantes por 30.

In [ ]:
# Rellenar duración

df_macro['duracion'] = df_macro['duracion'].fillna(30)

In [ ]:
# Revisar valores nulos después del relleno

nulos_macro = df_macro.isna().sum()

display(nulos_macro[nulos_macro > 0].sort_values(ascending=False))

,0


In [ ]:
from pathlib import Path

def carpeta_proyecto(nombre, marca):
    """Localiza los CSV en Jupyter, Visual Studio Code y Colab."""
    from pathlib import Path
    import json

    saltar = {
        ".config", "sample_data", "drive", ".git", "__pycache__",
        "node_modules", ".ipynb_checkpoints", "site-packages",
    }

    def directorio_notebook():
        try:
            return Path(__vsc_ipynb_file__).resolve().parent
        except NameError:
            pass
        try:
            from urllib.request import urlopen
            try:
                from ipykernel.connect import get_connection_file
            except ImportError:
                from ipykernel import get_connection_file
            conexion = Path(get_connection_file())
            kernel_id = conexion.name.replace("kernel-", "").replace(".json", "")
            servidores = list(conexion.parent.glob("jpserver-*.json"))[:3]
            for servidor in servidores:
                try:
                    info = json.loads(servidor.read_text(encoding="utf-8"))
                except (OSError, json.JSONDecodeError):
                    continue
                url = str(info.get("url", "")).rstrip("/")
                raiz = info.get("root_dir")
                if not url or not raiz:
                    continue
                api = url + "/api/sessions"
                token = info.get("token") or ""
                if token:
                    api += "?token=" + token
                try:
                    with urlopen(api, timeout=0.8) as respuesta:
                        sesiones = json.loads(respuesta.read().decode())
                except Exception:
                    continue
                for sesion in sesiones:
                    kid = (sesion.get("kernel") or {}).get("id")
                    if kid != kernel_id:
                        continue
                    cuaderno = sesion.get("notebook") or {}
                    relativo = cuaderno.get("path") or sesion.get("path")
                    if relativo:
                        return (Path(raiz) / relativo).resolve().parent
        except Exception:
            pass
        return None

    bases = []
    notebook = directorio_notebook()
    if notebook is not None:
        bases.append(notebook)
    bases.append(Path.cwd().resolve())
    content = Path("/content")
    if content.is_dir():
        bases.append(content)
        drive = content / "drive" / "MyDrive"
        if drive.is_dir():
            bases.append(drive)

    nombradas = []
    sueltas = []
    vistos = set()

    def considerar(directorio, aceptar_suelto):
        if directorio in vistos:
            return
        vistos.add(directorio)
        try:
            carpeta = directorio / nombre
            if (carpeta / marca).is_file():
                nombradas.append(carpeta)
                return
            if directorio.name == nombre and (directorio / marca).is_file():
                nombradas.append(directorio)
                return
            if aceptar_suelto and (directorio / marca).is_file():
                sueltas.append(directorio)
        except OSError:
            return

    def hijos(directorio, profundidad, limite):
        if profundidad < 0 or limite[0] <= 0 or not directorio.is_dir():
            return
        try:
            entradas = list(directorio.iterdir())
        except OSError:
            return
        for hijo in entradas:
            if not hijo.is_dir() or hijo.name.startswith(".") or hijo.name in saltar:
                continue
            limite[0] -= 1
            considerar(hijo, False)
            if nombradas or limite[0] <= 0:
                return
            hijos(hijo, profundidad - 1, limite)
            if nombradas or limite[0] <= 0:
                return

    for base in bases:
        considerar(base, True)
        if nombradas:
            break
        for padre in list(base.parents)[:6]:
            considerar(padre, False)
            if nombradas:
                break
        if nombradas:
            break
        profundidad = 3 if base.name in {"content", "MyDrive"} else 2
        hijos(base, profundidad, [400])
        if nombradas:
            break

    if nombradas:
        return nombradas[0]
    if sueltas:
        return sueltas[0]
    raise FileNotFoundError(
        "No se encuentra " + marca + ". "
        "Tiene que existir la carpeta '" + nombre + "' en el proyecto, "
        "o el archivo en el directorio de trabajo. En Colab puede estar junto a sample_data."
    )


def carpeta_datos():
    """Carpeta datos preprocesados, en Jupyter, Visual Studio Code o Colab."""
    return carpeta_proyecto("datos preprocesados", "ids_train.csv")

DATOS = carpeta_datos()
print("Datos:", DATOS)

# Cargar IDs de train y test

ids_train = pd.read_csv(DATOS / "ids_train.csv")['id_cancion']
ids_test = pd.read_csv(DATOS / "ids_test.csv")['id_cancion']

In [ ]:
# Crear train y test desde la tabla final

df_train = df_macro[df_macro['id_cancion'].isin(ids_train)].copy()
df_test = df_macro[df_macro['id_cancion'].isin(ids_test)].copy()

In [ ]:
print("df_train:", df_train.shape)
print("df_test:", df_test.shape)

df_train: (7990, 129)
df_test: (2000, 129)


In [ ]:
# Mostrar columnas disponibles

print(df_train.columns.tolist())

['id_cancion', 'filename_3s', 'length_3s', 'chroma_stft_mean_3s', 'chroma_stft_var_3s', 'rms_mean_3s', 'rms_var_3s', 'spectral_centroid_mean_3s', 'spectral_centroid_var_3s', 'spectral_bandwidth_mean_3s', 'spectral_bandwidth_var_3s', 'rolloff_mean_3s', 'rolloff_var_3s', 'zero_crossing_rate_mean_3s', 'zero_crossing_rate_var_3s', 'harmony_mean_3s', 'harmony_var_3s', 'perceptr_mean_3s', 'perceptr_var_3s', 'tempo_3s', 'mfcc1_mean_3s', 'mfcc1_var_3s', 'mfcc2_mean_3s', 'mfcc2_var_3s', 'mfcc3_mean_3s', 'mfcc3_var_3s', 'mfcc4_mean_3s', 'mfcc4_var_3s', 'mfcc5_mean_3s', 'mfcc5_var_3s', 'mfcc6_mean_3s', 'mfcc6_var_3s', 'mfcc7_mean_3s', 'mfcc7_var_3s', 'mfcc8_mean_3s', 'mfcc8_var_3s', 'mfcc9_mean_3s', 'mfcc9_var_3s', 'mfcc10_mean_3s', 'mfcc10_var_3s', 'mfcc11_mean_3s', 'mfcc11_var_3s', 'mfcc12_mean_3s', 'mfcc12_var_3s', 'mfcc13_mean_3s', 'mfcc13_var_3s', 'mfcc14_mean_3s', 'mfcc14_var_3s', 'mfcc15_mean_3s', 'mfcc15_var_3s', 'mfcc16_mean_3s', 'mfcc16_var_3s', 'mfcc17_mean_3s', 'mfcc17_var_3s', 'mfcc1

# Selección de variables

Se eliminan las columnas identificativas, de texto y la variable objetivo. El modelo se entrena únicamente con variables numéricas.

In [ ]:
# Separar variables de entrada y objetivo

columnas_no_modelo = [
    'id_cancion',
    'cancion',
    'cancion_base',
    'filename_3s',
    'filename_30s',
    'label_3s',
    'label_30s',
    'id_genero',
    'id_disco',
    'id_autor',
    'id_discografica'
]

columnas_no_modelo = [col for col in columnas_no_modelo if col in df_train.columns]

columnas_modelo = (
    df_train
    .drop(columns=columnas_no_modelo)
    .select_dtypes(include='number')
    .columns
    .tolist()
)

X_train = df_train[columnas_modelo].copy()
X_test = df_test[columnas_modelo].copy()

y_train = df_train['id_genero']
y_test = df_test['id_genero']

In [ ]:
# Revisar tamaño de los datos

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (7990, 118)
X_test: (2000, 118)
y_train: (7990,)
y_test: (2000,)


In [ ]:
# Revisar tipos de columnas

print(X_train.dtypes.value_counts())


float64    116
int64        2
Name: count, dtype: int64


# Construcción y entrenamiento del modelo

Primero se entrenó un Random Forest con una configuración inicial definida manualmente para obtener una referencia de rendimiento. Después se realizó una búsqueda de hiperparámetros para comprobar si una configuración optimizada podía mejorar ese resultado.

In [ ]:
# Cargar modelo y métricas

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
# Entrenar modelo Random Forest

modelo_rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

modelo_rf.fit(X_train, y_train)

RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42)

In [ ]:
# Crear predicciones

y_pred = modelo_rf.predict(X_test)

# Evaluación del modelo base

Se evalúa el modelo sobre el conjunto de test mediante accuracy, classification report y matriz de confusión.

In [ ]:
# Calcular accuracy

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

Accuracy: 0.804


El modelo obtiene un accuracy del 80,4% clasificando fragmentos de 3 segundos.

In [ ]:
# Mostrar informe de clasificación

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           1       0.82      0.82      0.82       200
           2       0.88      0.97      0.93       200
           3       0.78      0.84      0.81       200
           4       0.69      0.69      0.69       200
           5       0.75      0.77      0.76       200
           6       0.81      0.85      0.83       200
           7       0.89      0.85      0.87       200
           8       0.84      0.87      0.86       200
           9       0.78      0.79      0.78       200
          10       0.79      0.58      0.67       200

    accuracy                           0.80      2000
   macro avg       0.80      0.80      0.80      2000
weighted avg       0.80      0.80      0.80      2000



El modelo clasifica bastante bien la mayoría de géneros, pero tiene más dificultades con los géneros 4 y 10.

In [ ]:
# Crear matriz de confusión

matriz_confusion = confusion_matrix(y_test, y_pred)

print(matriz_confusion)

[[164   0  14   2   0   3   0   0  15   2]
 [  0 195   0   0   0   5   0   0   0   0]
 [  3   3 168   0   0   6   0   0   0  20]
 [  0   3  10 139  27   5  12   0   2   2]
 [ 10   0   0  20 154   0  10   0   0   6]
 [  0  20  10   0   0 170   0   0   0   0]
 [  0   0   0  23   0   0 170   0   7   0]
 [  0   0   0   6   6   0   0 174  14   0]
 [  0   0   0   0  10   0   0  32 158   0]
 [ 24   0  14  10   9  20   0   0   7 116]]


El modelo Random Forest obtiene un accuracy de 0.804, lo que indica que clasifica correctamente el 80,4% de los fragmentos de test. El classification report muestra un rendimiento equilibrado en la mayoría de géneros, destacando especialmente el género 2, con un f1-score de 0.93. Sin embargo, el modelo presenta más dificultad en los géneros 4 y 10, cuyos f1-score son 0.69 y 0.67 respectivamente. La matriz de confusión confirma estas diferencias, mostrando que el género 10 se confunde con varios géneros, especialmente con los géneros 1, 3 y 6.

# Búsqueda de hiperparámetros

Se utiliza RandomizedSearchCV para probar diferentes combinaciones de parámetros del Random Forest. La validación se realiza agrupando por id_cancion para evitar que fragmentos de una misma canción se mezclen entre entrenamiento y validación.

In [ ]:
# Crear grupos por canción

grupos_train = df_train['id_cancion']

In [ ]:
# Preparar búsqueda de mejores parámetros

from sklearn.model_selection import GroupKFold, RandomizedSearchCV

parametros_rf = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']
}

cv_grupos = GroupKFold(n_splits=5)

modelo_base_rf = RandomForestClassifier(
    random_state=42,
    n_jobs=-1
)

busqueda_rf = RandomizedSearchCV(
    estimator=modelo_base_rf,
    param_distributions=parametros_rf,
    n_iter=20,
    scoring='accuracy',
    cv=cv_grupos,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

In [ ]:
# Buscar mejores parámetros

busqueda_rf.fit(X_train, y_train, groups=grupos_train)

print("Mejores parámetros:")
print(busqueda_rf.best_params_)

print("Mejor accuracy en validación:")
print(busqueda_rf.best_score_)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Mejores parámetros:
{'n_estimators': 100, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': 30}
Mejor accuracy en validación:
0.80738423028786


# Evaluación del modelo ajustado

Una vez obtenidos los mejores parámetros, se evalúa el modelo ajustado sobre el conjunto de test. Esta evaluación permite comparar su rendimiento con el modelo base entrenado anteriormente.

Durante la validación interna dentro de train, la mejor combinación consiguió un accuracy medio del 80,73%.

In [ ]:
# Guardar mejor modelo encontrado

mejor_modelo_rf = busqueda_rf.best_estimator_
print(mejor_modelo_rf)

RandomForestClassifier(max_depth=30, n_jobs=-1, random_state=42)


In [ ]:
# Crear predicciones del mejor modelo

y_pred_mejor = mejor_modelo_rf.predict(X_test)

In [ ]:
# Comparar modelo base y modelo ajustado

accuracy_mejor = accuracy_score(y_test, y_pred_mejor)

print("Accuracy modelo base:", accuracy)
print("Accuracy modelo ajustado:", accuracy_mejor)

Accuracy modelo base: 0.804
Accuracy modelo ajustado: 0.7775


In [ ]:
# Mostrar informe del mejor modelo

print(classification_report(y_test, y_pred_mejor))

              precision    recall  f1-score   support

           1       0.77      0.77      0.77       200
           2       0.90      0.99      0.94       200
           3       0.70      0.79      0.74       200
           4       0.71      0.68      0.69       200
           5       0.70      0.70      0.70       200
           6       0.82      0.85      0.84       200
           7       0.88      0.84      0.86       200
           8       0.82      0.84      0.83       200
           9       0.72      0.76      0.74       200
          10       0.74      0.55      0.63       200

    accuracy                           0.78      2000
   macro avg       0.78      0.78      0.77      2000
weighted avg       0.78      0.78      0.77      2000



In [ ]:
# Crear matriz de confusión del mejor modelo

matriz_confusion_mejor = confusion_matrix(y_test, y_pred_mejor)

print(matriz_confusion_mejor)


[[153   0  21   0   0   7   0   0  17   2]
 [  0 198   0   0   0   2   0   0   0   0]
 [ 14   0 157   0   0  11   0   0   0  18]
 [  0   3  11 136  34   1   9   0   5   1]
 [ 10   0   0  22 141   0  10   0   0  17]
 [  1  19   9   0   0 171   0   0   0   0]
 [  0   0   0  14   8   0 169   0   9   0]
 [  0   0   0  10   1   0   0 168  21   0]
 [  0   0   0   0  10   0   0  38 152   0]
 [ 22   0  26  10   6  17   3   0   6 110]]


In [ ]:
# Mostrar matriz de confusión en tabla

etiquetas = sorted(y_test.unique())

df_matriz_confusion_mejor = pd.DataFrame(
    matriz_confusion_mejor,
    index=[f"Real {etiqueta}" for etiqueta in etiquetas],
    columns=[f"Predicho {etiqueta}" for etiqueta in etiquetas]
)

display(df_matriz_confusion_mejor)

,Predicho 1,Predicho 2,Predicho 3,Predicho 4,Predicho 5,Predicho 6,Predicho 7,Predicho 8,Predicho 9,Predicho 10
Real 1,153,0,21,0,0,7,0,0,17,2
Real 2,0,198,0,0,0,2,0,0,0,0
Real 3,14,0,157,0,0,11,0,0,0,18
Real 4,0,3,11,136,34,1,9,0,5,1
Real 5,10,0,0,22,141,0,10,0,0,17
Real 6,1,19,9,0,0,171,0,0,0,0
Real 7,0,0,0,14,8,0,169,0,9,0
Real 8,0,0,0,10,1,0,0,168,21,0
Real 9,0,0,0,0,10,0,0,38,152,0
Real 10,22,0,26,10,6,17,3,0,6,110


# Evaluación por canción

Como la tabla macro contiene varios fragmentos por canción, se agrupan las predicciones por id_cancion. La predicción final de cada canción se obtiene mediante votación mayoritaria entre sus fragmentos.

In [ ]:
# Guardar predicciones por fragmento

df_predicciones_fragmentos = df_test[['id_cancion', 'id_genero']].copy()

df_predicciones_fragmentos['genero_predicho'] = y_pred

display(df_predicciones_fragmentos.head())

,id_cancion,id_genero,genero_predicho
10,2,1,9
11,2,1,9
12,2,1,9
13,2,1,9
14,2,1,9


In [ ]:
# Crear predicción final por canción

df_predicciones_cancion = (
    df_predicciones_fragmentos
    .groupby('id_cancion')
    .agg(
        genero_real=('id_genero', 'first'),
        genero_predicho=('genero_predicho', lambda x: x.mode().iloc[0])
    )
    .reset_index()
)

display(df_predicciones_cancion.head())

,id_cancion,genero_real,genero_predicho
0,2,1,9
1,3,1,3
2,15,1,1
3,21,1,1
4,22,1,1


In [ ]:
# Evaluar predicción por canción

accuracy_cancion = accuracy_score(
    df_predicciones_cancion['genero_real'],
    df_predicciones_cancion['genero_predicho']
)

print("Accuracy por fragmento:", accuracy)
print("Accuracy por canción:", accuracy_cancion)

Accuracy por fragmento: 0.804
Accuracy por canción: 0.81


In [ ]:
# Mostrar informe por canción

print(classification_report(
    df_predicciones_cancion['genero_real'],
    df_predicciones_cancion['genero_predicho']
))

              precision    recall  f1-score   support

           1       0.81      0.85      0.83        20
           2       0.91      1.00      0.95        20
           3       0.77      0.85      0.81        20
           4       0.70      0.70      0.70        20
           5       0.73      0.80      0.76        20
           6       0.81      0.85      0.83        20
           7       0.89      0.85      0.87        20
           8       0.85      0.85      0.85        20
           9       0.80      0.80      0.80        20
          10       0.85      0.55      0.67        20

    accuracy                           0.81       200
   macro avg       0.81      0.81      0.81       200
weighted avg       0.81      0.81      0.81       200



In [ ]:
# Crear matriz de confusión por canción

matriz_confusion_cancion = confusion_matrix(
    df_predicciones_cancion['genero_real'],
    df_predicciones_cancion['genero_predicho']
)

print(matriz_confusion_cancion)

[[17  0  2  0  0  0  0  0  1  0]
 [ 0 20  0  0  0  0  0  0  0  0]
 [ 0  0 17  0  0  1  0  0  0  2]
 [ 0  0  1 14  3  1  1  0  0  0]
 [ 1  0  0  2 16  0  1  0  0  0]
 [ 0  2  1  0  0 17  0  0  0  0]
 [ 0  0  0  2  0  0 17  0  1  0]
 [ 0  0  0  1  1  0  0 17  1  0]
 [ 0  0  0  0  1  0  0  3 16  0]
 [ 3  0  1  1  1  2  0  0  1 11]]


# Conclusiones

El modelo Random Forest base obtuvo un accuracy de 0.804 sobre los fragmentos del conjunto de test, clasificando correctamente el 80,4% de los fragmentos analizados.

El classification report muestra un rendimiento correcto en la mayoría de géneros, destacando especialmente el género 2. Sin embargo, el modelo presenta más dificultad en los géneros 4 y 10.

Posteriormente se realizó una búsqueda de hiperparámetros mediante RandomizedSearchCV. Aunque la mejor combinación obtuvo un accuracy medio de 0.8073 durante la validación interna, el modelo ajustado alcanzó un accuracy de 0.7775 sobre el conjunto de test, inferior al modelo base. Por este motivo, se mantiene el modelo base como mejor opción.

Finalmente, al trabajar con una tabla macro formada por fragmentos de canciones, se realizó una evaluación adicional por canción completa mediante votación mayoritaria entre fragmentos. En esta evaluación, el modelo obtuvo un accuracy de 0.81, mejorando ligeramente el resultado obtenido por fragmento y ofreciendo una predicción más alineada con el objetivo real del problema.